# PAEC Prefix Training (Colab T4)

Train two perspective prefixes (self + partner) on ToMi + ExploreToM data.
Expected runtime: ~1-1.5 hours on T4 GPU.

In [ ]:
# Install dependencies
!pip install -q torch transformers peft datasets bitsandbytes accelerate tqdm pyyaml scipy scikit-learn

# Mount Google Drive for checkpoints
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone the project
!git clone https://github.com/vualidon/tom_agent_collab.git /content/tom_agent_collab 2>/dev/null || echo 'Already cloned'
%cd /content/tom_agent_collab

# Clone external repos
!git clone https://github.com/facebookresearch/ToMi.git external/ToMi 2>/dev/null || echo 'ToMi already cloned'

In [ ]:
# Generate ToMi dataset
%cd external/ToMi
!python main.py
%cd /content/tom_agent_collab

In [ ]:
import torch
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

from src.config import PAECConfig
from src.data.prefix_training_data import combine_training_data

config = PAECConfig.from_yaml('configs/default.yaml')
train_data, tomi_test = combine_training_data('external/ToMi', seed=42)
print(f'\nTraining examples: {len(train_data)}')
print(f'ToMi test examples: {len(tomi_test)}')

In [ ]:
# Train self-perspective prefix
from src.models.model_loader import load_base_model
from src.models.perspective_prefix import train_prefix

base_model, tokenizer = load_base_model(config)

self_prefix_model = train_prefix(
    config=config,
    train_data=train_data,
    perspective='self',
    output_dir='/content/drive/MyDrive/paec_checkpoints/prefix_self',
    tokenizer=tokenizer,
    base_model=base_model,
)

In [ ]:
# Train partner-perspective prefix (reload base model)
import gc
del self_prefix_model
gc.collect()
torch.cuda.empty_cache()

base_model, tokenizer = load_base_model(config)

partner_prefix_model = train_prefix(
    config=config,
    train_data=train_data,
    perspective='partner',
    output_dir='/content/drive/MyDrive/paec_checkpoints/prefix_partner',
    tokenizer=tokenizer,
    base_model=base_model,
)

In [ ]:
# Verify prefix diversity
import gc
del partner_prefix_model, base_model
gc.collect()
torch.cuda.empty_cache()

from src.models.model_loader import load_model_with_prefix

model1, _ = load_model_with_prefix(config, '/content/drive/MyDrive/paec_checkpoints/prefix_self')
emb1 = model1.get_prompt(batch_size=1).detach().cpu().flatten()
del model1; gc.collect(); torch.cuda.empty_cache()

model2, _ = load_model_with_prefix(config, '/content/drive/MyDrive/paec_checkpoints/prefix_partner')
emb2 = model2.get_prompt(batch_size=1).detach().cpu().flatten()
del model2; gc.collect(); torch.cuda.empty_cache()

cosine_sim = torch.nn.functional.cosine_similarity(emb1.unsqueeze(0), emb2.unsqueeze(0))
print(f'Prefix cosine similarity: {cosine_sim.item():.4f}')
print('(Lower = more distinct perspectives)')